In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib.pyplot as plt
import os


In [ ]:
DATA_DIR = "./dataset" # change path if needed
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
DATA_DIR,
validation_split=0.2,
subset="training",
seed=42,
image_size=IMG_SIZE,
batch_size=BATCH_SIZE
)

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
DATA_DIR,
validation_split=0.2,
subset="validation",
seed=42,
image_size=IMG_SIZE,
batch_size=BATCH_SIZE
)


In [ ]:
class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


In [ ]:
data_augmentation = models.Sequential([
layers.RandomFlip("horizontal"),
layers.RandomRotation(0.1),
layers.RandomZoom(0.1)
])

In [ ]:
base_model = MobileNetV2(
input_shape=IMG_SIZE + (3,),
include_top=False,
weights="imagenet"
)

In [ ]:
base_model.trainable = False


In [ ]:
model = models.Sequential([
layers.Input(shape=IMG_SIZE + (3,)),
data_augmentation,
layers.Lambda(preprocess_input),
base_model,
layers.GlobalAveragePooling2D(),
layers.Dense(128, activation="relu"),
layers.Dropout(0.3),
layers.Dense(num_classes, activation="softmax")
])


In [ ]:

model.summary()


In [ ]:
model.compile(
optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
loss="sparse_categorical_crossentropy",
metrics=["accuracy"]
)

In [ ]:
history = model.fit(
train_ds,
validation_data=val_ds,
epochs=EPOCHS
)

In [ ]:

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')

plt.show()


In [ ]:
model.save("food_classifier_tf")
print("Model saved as food_classifier_tf")

In [2]:
import kagglehub
import os

path = kagglehub.dataset_download("kmader/food41")
KAGGLE_DATASET_PATH = path

Resuming download from 208666624 bytes (5485141482 bytes left)...
Resuming download from https://www.kaggle.com/api/v1/datasets/download/kmader/food41?dataset_version_number=5 (208666624/5693808106) bytes left.
Resuming download from https://www.kaggle.com/api/v1/datasets/download/kmader/food41?dataset_version_number=5 (208666624/5693808106) bytes left.


  4%|▍         | 205M/5.30G [00:08<2:05:34, 727kB/s]



KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
import json

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
DATASET_PATH = KAGGLE_DATASET_PATH

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001


In [ ]:

MODEL_SAVE_PATH = "food_recognition_model.h5"
LABELS_SAVE_PATH = "food_labels.json"

In [ ]:
if not os.path.exists(DATASET_PATH):
    raise ValueError(f"Dataset path not found: {DATASET_PATH}")

food_categories = sorted([d for d in os.listdir(DATASET_PATH) 
                         if os.path.isdir(os.path.join(DATASET_PATH, d)) 
                         and not d.startswith('.')])

labels_dict = {i: category for i, category in enumerate(food_categories)}
with open(LABELS_SAVE_PATH, 'w') as f:
    json.dump(labels_dict, f, indent=2)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

test_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

validation_generator = test_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

In [ ]:
def create_model(num_classes):
    base_model = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    
    base_model.trainable = False
    
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

num_classes = train_generator.num_classes
model, base_model = create_model(num_classes)



In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')]
)

In [ ]:
checkpoint = ModelCheckpoint(
    MODEL_SAVE_PATH,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

callbacks = [checkpoint, early_stop, reduce_lr]

In [ ]:
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False


In [ ]:

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE/10),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')]
)


In [ ]:
history_fine = model.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)